# 05 — Diagnostics & Validation

Aggregate out-of-sample predictions and produce the Phase 1 PASS/FAIL decision.

In [ ]:
import torch, sys
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime → T4"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE  = '/content/drive/MyDrive/yeniBot'
DATA_DIR    = f'{DRIVE_BASE}/data'
CHECKPT_DIR = f'{DRIVE_BASE}/checkpoints'
os.makedirs(CHECKPT_DIR, exist_ok=True)

In [ ]:
import os, sys
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = '/content/yenibot_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print('After git pull, use Runtime → Restart session before trusting changed imports.')

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import yaml
with open(f'{REPO_DIR}/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['data_dir'] = DATA_DIR
cfg['paths']['checkpoint_dir'] = CHECKPT_DIR

In [ ]:
import glob
import pandas as pd
from yenibot.diagnostics import (
    calibrate_test_probabilities_from_val,
    calibration_table,
    fold_diagnostics,
    good_bad_feature_audit,
    mtf_leakage_diagnostics,
    phase1_report,
    stationarity_policy_diagnostics,
    regime_diagnostics,
    threshold_diagnostics,
)

pred_path = f'{CHECKPT_DIR}/predictions_all.parquet'
if os.path.exists(pred_path):
    predictions = pd.read_parquet(pred_path)
else:
    parts = [pd.read_parquet(path) for path in sorted(glob.glob(f'{CHECKPT_DIR}/predictions_fold_*.parquet'))]
    predictions = pd.concat(parts, ignore_index=True)

test_predictions = predictions[predictions['split'] == 'test'].copy()
report = phase1_report(test_predictions, cfg)
print('PHASE 1:', 'PASS — Ready for Phase 2' if report['passed'] else 'FAIL — Do NOT proceed')
print(report)
calibration = calibration_table(test_predictions['label'], test_predictions['prob_long'], bins=cfg['validation']['calibration_bins'])
fold_metrics = fold_diagnostics(test_predictions)
regime_metrics = regime_diagnostics(test_predictions)
threshold_metrics = threshold_diagnostics(predictions)
mtf_leakage = mtf_leakage_diagnostics(test_predictions)
feature_audit = good_bad_feature_audit(test_predictions, fold_metrics)
model_paths = sorted(glob.glob(f'{CHECKPT_DIR}/model_fold_*.pt'))
saved_feature_columns = []
if model_paths:
    try:
        checkpoint = torch.load(model_paths[0], map_location='cpu', weights_only=False)
    except TypeError:
        checkpoint = torch.load(model_paths[0], map_location='cpu')
    saved_feature_columns = list(checkpoint.get('feature_columns', []))
stationarity_policy = stationarity_policy_diagnostics(saved_feature_columns, cfg)
calibrated_test_predictions, calibrated_report, calibrated_calibration = calibrate_test_probabilities_from_val(predictions, cfg)
display(calibration)
display(fold_metrics)
display(regime_metrics)
display(threshold_metrics)
display(mtf_leakage)
display(stationarity_policy)
display(feature_audit.head(25))
print('CALIBRATED PHASE 1:', 'PASS' if calibrated_report['passed'] else 'FAIL')
print(calibrated_report)
display(calibrated_calibration)

In [ ]:
import matplotlib.pyplot as plt
rank_by_fold = pd.Series(report['rank_ic_by_fold']).sort_index()
rank_by_fold.plot(kind='bar', title='Test Rank IC by fold')
plt.axhline(cfg['validation']['target_rank_ic'], color='green', linestyle='--')
plt.axhline(0, color='black', linewidth=1)
plt.show()

calibration.plot(x='mean_prob_long', y='actual_long_rate', kind='scatter', title='Calibration')
plt.plot([0, 1], [0, 1], linestyle='--', color='black')
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
threshold = 0.5
for regime_col in [c for c in test_predictions.columns if c.startswith('regime_prob_')]:
    regime = int(regime_col.rsplit('_', 1)[-1])
    subset = test_predictions[test_predictions[[c for c in test_predictions.columns if c.startswith('regime_prob_')]].idxmax(axis=1) == regime_col]
    if len(subset):
        print('Regime', regime, confusion_matrix(subset['label'], subset['prob_long'] >= threshold))

In [ ]:
from yenibot.diagnostics import load_fold_model, extract_embeddings, tsne_embeddings

fold_id = int(fold_metrics.sort_values('rank_ic', ascending=False).iloc[0]['fold'])
model, saved_feature_columns = load_fold_model(f'{CHECKPT_DIR}/model_fold_{fold_id:03d}.pt')
fold_test = test_predictions[(test_predictions['fold'] == fold_id) & (test_predictions['split'] == 'test')].copy()
embeddings = extract_embeddings(model, fold_test, saved_feature_columns, seq_len=cfg['model']['seq_len'])
tsne = tsne_embeddings(embeddings, random_state=cfg['project']['random_seed'])
tsne.plot.scatter(x='tsne_x', y='tsne_y', c='label', colormap='viridis', title=f'Fold {fold_id} OOS latent t-SNE')
plt.show()

In [ ]:
from yenibot.diagnostics import permutation_importance_rank_ic

selected_importance_folds = sorted(set([
    int(fold_metrics.sort_values('rank_ic', ascending=False).iloc[0]['fold']),
    int(fold_metrics.sort_values('rank_ic', ascending=True).iloc[0]['fold']),
    int(fold_metrics.iloc[(fold_metrics['rank_ic'].abs()).argsort().iloc[0]]['fold']),
]))
importance_parts = []
for importance_fold in selected_importance_folds:
    fold_model, fold_feature_columns = load_fold_model(f'{CHECKPT_DIR}/model_fold_{importance_fold:03d}.pt')
    fold_part = test_predictions[(test_predictions['fold'] == importance_fold) & (test_predictions['split'] == 'test')].copy()
    fold_importance = permutation_importance_rank_ic(
        fold_model,
        fold_part,
        fold_feature_columns,
        seq_len=cfg['model']['seq_len'],
        n_repeats=1,
    )
    fold_importance['fold'] = importance_fold
    fold_importance['fold_rank_ic'] = float(fold_metrics.loc[fold_metrics['fold'] == importance_fold, 'rank_ic'].iloc[0])
    importance_parts.append(fold_importance)
importance = pd.concat(importance_parts, ignore_index=True)
importance_summary = importance.groupby('feature', as_index=False).agg(
    mean_rank_ic_drop=('rank_ic_drop', 'mean'),
    min_rank_ic_drop=('rank_ic_drop', 'min'),
    max_rank_ic_drop=('rank_ic_drop', 'max'),
    folds=('fold', lambda x: ','.join(map(str, sorted(x))))
).sort_values('mean_rank_ic_drop', ascending=False)
display(importance_summary.head(25))
display(importance.head(25))

In [ ]:
from yenibot.diagnostics import write_phase1_diagnostic_bundle

REPORT_DIR = f'{DRIVE_BASE}/reports'
zip_path = write_phase1_diagnostic_bundle(
    output_dir=REPORT_DIR,
    report=report,
    predictions=test_predictions,
    calibration=calibration,
    fold_metrics=fold_metrics,
    regime_metrics=regime_metrics,
    importance=importance if 'importance' in globals() else None,
    tsne=tsne if 'tsne' in globals() else None,
    calibrated_report=calibrated_report,
    calibrated_calibration=calibrated_calibration,
    calibrated_predictions=calibrated_test_predictions,
    threshold_metrics=threshold_metrics,
    mtf_leakage=mtf_leakage,
    feature_audit=feature_audit,
    stationarity_policy=stationarity_policy if 'stationarity_policy' in globals() else None,
    config=cfg,
)
print(f'Saved diagnostic bundle: {zip_path}')

In [ ]:
from google.colab import runtime
runtime.unassign()